<h1>Semi-Supervised Learning & Label Spreading</h1>

In semi-supervised learning only **some** of the samples have labels. Semi-supervised
algorithms sit between supervised and unsupervised learning: they use the labelled
samples *and* the shape of the unlabelled data to estimate the missing labels.

**Label Spreading** builds a graph connecting similar data points, then lets the known
labels spread across the graph to their unlabelled neighbours — like drops of dye
spreading through water. (Technically: it minimises a loss function with regularisation,
using edge weights from the normalised graph Laplacian matrix.)

In this notebook we use the classic IRIS dataset: 150 flowers, 4 measurements each,
3 species — then hide most of the labels and see how well they can be recovered.

<h1>Step 1 : Import Libraries</h1>

In [ ]:

# import the libraries we need:
# numpy               -> random numbers and array handling
# datasets            -> sklearn's built-in datasets (includes IRIS)
# LabelSpreading      -> the semi-supervised algorithm
# LogisticRegression  -> a supervised model to compare against
# accuracy_score, confusion_matrix -> for honest evaluation
import numpy as np
from sklearn import datasets
from sklearn.semi_supervised import LabelSpreading
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

<h1>Step 2 : Import Dataset</h1>
IRIS ships inside scikit-learn, so there is no file to download — this always works.

In [ ]:
# load the IRIS dataset
iris = datasets.load_iris()

# quick orientation: how many flowers, how many measurements, which species
print("data shape:", iris.data.shape)          # (150 flowers, 4 measurements)
print("species:", list(iris.target_names))     # 0=setosa, 1=versicolor, 2=virginica

<h1>Step 3 : Create the random number generator and the model</h1>
We create a seeded random number generator (so every run — and every student —
gets exactly the same result), and an instance of the LabelSpreading model.

In [ ]:

# a random number generator with a fixed seed — same "random" choices every run
rand = np.random.RandomState(0)

# the Label Spreading model, with its default settings
label_spreading_model = LabelSpreading()

<h1>Step 3.1: Hiding the labels</h1>
Now we hide roughly 70% of the labels. scikit-learn's convention for
"this sample is unlabelled" is the value **-1** — the model treats every
-1 as a label it must work out for itself.

In [ ]:
# draw a random number 0-1 for each flower; True (about 70% of them) = hide its label
random_unlabeled_samples = rand.rand(len(iris.target)) <= 0.7

# copy the true labels, then overwrite the hidden ones with -1
# (we keep iris.target untouched — we need the truth later for marking)
un_labelled = np.copy(iris.target)
un_labelled[random_unlabeled_samples] = -1

# how did the split land?
print("total flowers:", len(un_labelled))
print("labels hidden:", random_unlabeled_samples.sum())
print("labels kept:  ", (~random_unlabeled_samples).sum())

<h1>Step 3.2 : Inspect the labels</h1>
Every -1 below is a flower the model must label for itself. The scattered
0s, 1s and 2s are the few flowers that kept their species label.

In [ ]:
# print the label array the model will be given
print(un_labelled)

<h1>Step 4 : Fit the model and mark it honestly</h1>
We fit Label Spreading on all 150 flowers — the few labelled ones and the many
-1s — and let it predict a species for every flower.

**Important:** to measure accuracy we compare the predictions against the TRUE
labels we kept aside in `iris.target`. Comparing a model's predictions against
its own predictions would always give 100% and tell us nothing.

In [ ]:
# fit the model: it sees the measurements for all flowers,
# but labels for only 39 of them
label_spreading_model.fit(iris.data, un_labelled)

# predict a species for every flower
pred_lb = label_spreading_model.predict(iris.data)

# mark against the TRUE labels
overall = accuracy_score(iris.target, pred_lb)
print(f"Accuracy on all 150 flowers: {overall:.1%}")

# the real test: the flowers whose labels were hidden
hidden = random_unlabeled_samples
hidden_acc = accuracy_score(iris.target[hidden], pred_lb[hidden])
print(f"Accuracy on the {hidden.sum()} hidden flowers: {hidden_acc:.1%}",
      f"({(iris.target[hidden] == pred_lb[hidden]).sum()} of {hidden.sum()} recovered)")

<h1>Step 5: A fair comparison with supervised learning</h1>
How good is 92.8%? To judge it we need a fair opponent. A supervised model in the
same situation would have **only the 39 labelled flowers** to learn from — it cannot
use the unlabelled ones at all. So: train Logistic Regression on those 39, then have
both models label the same 111 hidden flowers, marked against the truth.

In [ ]:

# train Logistic Regression on ONLY the flowers that kept their labels
# (max_iter=1000 simply gives the solver plenty of room to converge)
supervised = LogisticRegression(random_state=0, max_iter=1000)
supervised.fit(iris.data[~hidden], iris.target[~hidden])

# both models label the same hidden flowers
lr_pred = supervised.predict(iris.data[hidden])
lr_acc = accuracy_score(iris.target[hidden], lr_pred)

print(f"Label Spreading  on the hidden flowers: {hidden_acc:.1%}")
print(f"Logistic Regression on the hidden flowers: {lr_acc:.1%}")

# confusion matrices (rows = actual species, columns = predicted)
print("\nLabel Spreading:")
print(confusion_matrix(iris.target[hidden], pred_lb[hidden]))
print("\nLogistic Regression:")
print(confusion_matrix(iris.target[hidden], lr_pred))

<h1>Step 6: So when does semi-supervised learning win?</h1>
With 39 labelled flowers, supervised learning edged ahead — IRIS is an easy dataset,
and 39 labels are plenty. Semi-supervised learning earns its keep when labels are
**scarce**: labelling data is often the expensive part of a real project (imagine
paying a botanist to identify each flower). Let's repeat the experiment hiding 90%
of the labels, so only about 14 flowers keep one.

In [ ]:
# same experiment, but hide 90% of the labels this time
rand2 = np.random.RandomState(0)
scarce = rand2.rand(len(iris.target)) <= 0.9

un_labelled_scarce = np.copy(iris.target)
un_labelled_scarce[scarce] = -1
print("labels kept this time:", (~scarce).sum())

# Label Spreading: learns from the few labels AND the shape of all the data
ls2 = LabelSpreading()
ls2.fit(iris.data, un_labelled_scarce)
ls2_acc = accuracy_score(iris.target[scarce], ls2.predict(iris.data)[scarce])

# Logistic Regression: can only learn from the few labelled flowers
lr2 = LogisticRegression(random_state=0, max_iter=1000)
lr2.fit(iris.data[~scarce], iris.target[~scarce])
lr2_acc = accuracy_score(iris.target[scarce], lr2.predict(iris.data[scarce]))

print(f"Label Spreading  on the hidden flowers: {ls2_acc:.1%}")
print(f"Logistic Regression on the hidden flowers: {lr2_acc:.1%}")

<h1>What did we learn?</h1>

- **-1 means "unlabelled".** That is scikit-learn's convention for semi-supervised
  learning: the model treats every -1 as a label to work out for itself.
- **Mark against the truth, never against yourself.** Scoring a model on its own
  predictions gives 100% every time and means nothing. We kept the true labels
  aside and marked against those.
- **With 39 labels (70% hidden):** Label Spreading recovered 92.8% of the hidden
  labels, but plain Logistic Regression trained on the same 39 scored 95.5% —
  IRIS is easy, and 39 labels are plenty for a supervised model.
- **With 14 labels (90% hidden):** the order flipped — Label Spreading 93.4% vs
  Logistic Regression 91.2%. The unlabelled flowers carry real information about
  the shape of the data, and semi-supervised learning is how you use it.
- **The rule of thumb:** the scarcer (more expensive) your labels, the more
  semi-supervised learning has to offer.